# 基本索引与切片

学习目标：按位置提取和修改数组区域，判断结果形状，并识别切片与原数组共享数据的影响。

前置知识：Python 索引与切片、数组形状与轴、dtype。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用首次导入的 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按位置读取元素

要查看一组测量中的首次和末次读数，可以分别使用索引 0 和 -1。索引从 0 开始，负索引从末尾向前计数。

下面 readings 是形状 (4,) 的一维数组。索引 1 读取第二个元素，得到带有原 dtype 的 NumPy 标量。

In [1]:
import numpy as np

readings = np.array([18, 20, 23, 21], dtype=np.int16)

print(readings[0], readings[-1])  # 18 21，首次与末次读数。
print(readings[1], readings[-2])  # 20 23，第二个与倒数第二个读数。
print(readings[1].dtype)  # int16，单个数值仍保留元素类型。

18 21
20 23
int16


## 2 切片与步长

切片用 readings[start:stop:step] 表示一段选择。readings 是被选择的数组，start 是起始索引，stop 是不包含的停止边界，step 是非零步长，省略时为 1。

正步长从前向后选择；省略起止边界时选择到相应端点。冒号 : 选择整个轴，::2 每隔一个位置取一个值，::-1 反向选择。

In [2]:
readings = np.array([10, 11, 12, 13, 14, 15], dtype=np.int16)

print(readings[1:4])  # [11 12 13]，包含索引 1，不包含索引 4。
print(readings[:3], readings[3:])  # [10 11 12] 与 [13 14 15]。
print(readings[::2])  # [10 12 14]，索引为 0、2、4。
print(readings[::-1])  # [15 14 13 12 11 10]。
print(readings[4:1:-1])  # [14 13 12]，反向取索引 4、3、2，不含 1。
print(readings[-3:])  # [13 14 15]，最后三个值。
print(readings[1:4].shape, readings[1:4].dtype)  # (3,) int16。

[11 12 13]
[10 11 12] [13 14 15]
[10 12 14]
[15 14 13 12 11 10]
[14 13 12]
[13 14 15]
(3,) int16


## 3 二维区域与维数

### 3.1 按行列选择

二维数组用逗号分隔各轴的选择。下面表格的形状为 (3, 4)，行表示三次观测，列表示四个传感器。

table[1, 2] 读取第二次观测的第三个传感器；table[1:3, 1:4] 取后两次观测、后三个传感器构成的矩形区域。

In [3]:
table = np.array([
    [11, 12, 13, 14],
    [21, 22, 23, 24],
    [31, 32, 33, 34],
], dtype=np.int16)
region = table[1:3, 1:4]

print(table[1, 2], table[-1, -1])  # 23 34。
print(region)  # 两行分别为 [22 23 24]、[32 33 34]。
print(region.shape, region.dtype)  # (2, 3) int16。
print(table[:, 0])  # [11 21 31]，所有观测中的第一个传感器。

23 34
[[22 23 24]
 [32 33 34]]
(2, 3) int16
[11 21 31]


### 3.2 整数索引与切片的区别

沿用上面的 table。table[1, :] 和 table[1:2, :] 选出同一行的四个数，结果却不具有相同形状：整数索引移除行轴，切片保留行轴，即使它的长度只剩 1。

![table 的第二行同时进入一维结果和一行四列的二维结果，值相同但保留的轴不同。](image/illustration/03-01-index-axis-retention.svg)

图中一维结果横向排放只是为了阅读，并不表示它自带行轴。判断维数要看索引是否保留轴，不看屏幕上打印了几行。

下面同时比较行选择与列选择，逐项核对 shape。省略末尾选择时，末尾各轴按完整切片处理，因此 table[1] 与 table[1, :] 相同。

In [4]:
row = table[1, :]
row_table = table[1:2, :]
column = table[:, 2]
column_table = table[:, 2:3]

print(row, row.shape)  # [21 22 23 24] (4,)。
print(row_table, row_table.shape)  # 一行四列，(1, 4)。
print(column, column.shape)  # [13 23 33] (3,)。
print(column_table, column_table.shape)  # 三行一列，(3, 1)。
print(table[1])  # [21 22 23 24]，与 row 相同。

[21 22 23 24] (4,)
[[21 22 23 24]] (1, 4)
[13 23 33] (3,)
[[13]
 [23]
 [33]] (3, 1)
[21 22 23 24]


## 4 三维选择与省略号

下面 batches 的形状为 (2, 3, 4)，三个轴依次表示批次、观测、传感器。batches[1, :2, 1:4] 选择第二批的前两次观测和后三个传感器，结果保留观测与传感器两个轴。

省略号 ... 补足所需的完整切片；在这里 batches[..., -1] 等价于 batches[:, :, -1]，提取所有批次、所有观测的末个传感器。一次索引最多使用一个省略号。

In [5]:
batches = np.array([
    [[11, 12, 13, 14], [21, 22, 23, 24], [31, 32, 33, 34]],
    [[111, 112, 113, 114], [121, 122, 123, 124], [131, 132, 133, 134]],
], dtype=np.int16)
region = batches[1, :2, 1:4]
last_sensor = batches[..., -1]

print(region)  # [112 113 114]、[122 123 124] 两行。
print(region.shape, region.dtype)  # (2, 3) int16。
print(last_sensor)  # 两行分别为 [14 24 34]、[114 124 134]。
print(last_sensor.shape)  # (2, 3)，批次轴和观测轴保留。
print(batches[1:2, :2, 1:4].shape)  # (1, 2, 3)，改用切片可保留批次轴。

[[112 113 114]
 [122 123 124]]
(2, 3) int16
[[ 14  24  34]
 [114 124 134]]
(2, 3)
(1, 2, 3)


## 5 用 newaxis 增加轴

一维读数本身没有“行”或“列”的方向。需要一行多列或多行一列的二维表示时，可以在索引中放入 np.newaxis，在该位置增加一个长度为 1 的轴。

np.newaxis 是 None 的别名，两种写法作用相同。增加轴不增加元素数量。

In [6]:
readings = np.array([18, 20, 22], dtype=np.int16)
row = readings[np.newaxis, :]
column = readings[:, np.newaxis]

print(row, row.shape)  # [[18 20 22]] (1, 3)。
print(column, column.shape)  # 三行一列，(3, 1)。
print(readings[:, None].shape)  # (3, 1)，与使用 np.newaxis 相同。
print(row.size, column.size, column.dtype)  # 3 3 int16。

[[18 20 22]] (1, 3)
[[18]
 [20]
 [22]] (3, 1)
(3, 1)
3 3 int16


## 6 切片赋值

### 6.1 修改选定区域

切片放在等号左侧可以修改原数组的选定位置。可以赋一个标量统一填充，也可以赋同形数据逐项替换；切片赋值不会扩展数组。

下面表格形状为 (3, 4)，行表示观测，列表示传感器。先把第一个传感器的读数统一设为 0，再替换后两次观测中第二、第三个传感器的读数。

In [7]:
table = np.array([
    [11, 12, 13, 14],
    [21, 22, 23, 24],
    [31, 32, 33, 34],
], dtype=np.int16)

table[:, 0] = 0
table[1:3, 1:3] = [[50, 51], [60, 61]]

print(table)  # 三行为 [0 12 13 14]、[0 50 51 24]、[0 60 61 34]。
print(table.shape, table.dtype)  # (3, 4) int16，形状和 dtype 不变。

[[ 0 12 13 14]
 [ 0 50 51 24]
 [ 0 60 61 34]]
(3, 4) int16


### 6.2 赋值时的类型与形状

赋入的值要转换为目标数组的 dtype。给整数数组赋小数，不会自动把整个数组改成浮点类型。

除了同形输入，NumPy 也允许符合广播规则的形状；本章先使用标量或同形数据。给长度为 2 的切片赋三个值不符合要求，会触发 ValueError。

In [8]:
counts = np.array([10, 20, 30], dtype=np.int16)
counts[:2] = [1.75, 2.25]
print(counts, counts.dtype)  # [ 1  2 30] int16，小数部分丢失。

# 预期 ValueError：三个值不能赋给长度为 2 的目标切片。
counts[:2] = [5, 6, 7]

[ 1  2 30] int16


ValueError: could not broadcast input array from shape (3,) into shape (2,)

In [9]:
print(counts.shape)  # 仍为 (3,)，赋值没有扩展数组。

(3,)


## 7 切片与原数组共享数据

基本切片返回视图（view），它通过另一个数组对象访问原来的数据。修改视图中的元素，会改变原数组对应的位置；原数组的修改也会在视图中体现。

下面用 np.shares_memory() 检查两个小数组是否有共享的元素内存，再通过修改验证影响范围。需要独立修改区域时，使用 copy() 复制该区域的数据。

In [10]:
table = np.array([[10, 11, 12], [20, 21, 22]], dtype=np.int16)
region = table[:, 1:3]
independent = region.copy()

print(region.shape, region.dtype)  # (2, 2) int16。
print(np.shares_memory(table, region))  # True，基本切片共享数据。
print(np.shares_memory(table, independent))  # False，复制后独立存储。

region[0, 0] = 99
table[1, 2] = 88
independent[0, 0] = -1

print(table)  # 两行为 [10 99 12]、[20 21 88]，只有对应位置变化。
print(region)  # 两行为 [99 12]、[21 88]，能看到原数组的修改。
print(independent)  # 两行为 [-1 12]、[21 22]，不受共享修改影响。

(2, 2) int16
True
False
[[10 99 12]
 [20 21 88]]
[[99 12]
 [21 88]]
[[-1 12]
 [21 22]]


## 8 越界与空切片

整数索引必须对应已有位置，越界会触发 IndexError。切片边界可以超过轴长，实际只保留范围内的位置；没有选中元素时得到空数组。

空切片仍保留切片所在的轴。下面 table[3:3, :] 的形状是 (0, 4)，表示没有观测行，但传感器轴仍有四个位置。

In [11]:
table = np.array([
    [11, 12, 13, 14],
    [21, 22, 23, 24],
    [31, 32, 33, 34],
], dtype=np.int16)

# 预期 IndexError：行轴长度为 3，位置 3 已越界。
print(table[3, 0])

IndexError: index 3 is out of bounds for axis 0 with size 3

In [12]:
print(table[1:10, 0])  # [21 31]，切片终点越界不要求读取不存在的位置。
empty = table[3:3, :]
print(empty.shape, empty.size, empty.dtype)  # (0, 4) 0 int16。
print(table[2:1, :].shape)  # (0, 4)，默认正步长不能从 2 走到更小的 1。

[21 31]
(0, 4) 0 int16
(0, 4)


## 本章小结

（1）每个轴分别写索引或切片，先说明轴含义，再判断选择的位置。

（2）整数索引移除对应轴，切片保留轴；newaxis 增加长度为 1 的轴，省略号补足完整切片。

（3）基本切片共享原数组数据；修改前应判断是否需要 copy()。

（4）赋值受到目标 shape 与 dtype 的约束。整数越界报错，空切片则是合法的空数组。

## 练习

（1）下面数组的行表示观测，列表示传感器。提取最后两次观测的第一个和第三个传感器，用基本切片一次完成，再打印值、shape 和 dtype。

In [13]:
table = np.array([
    [11, 12, 13, 14],
    [21, 22, 23, 24],
    [31, 32, 33, 34],
], dtype=np.int16)

# 在此填写切片；提示：传感器轴可用步长 2。
# 检查：两行为 [21 23]、[31 33]，shape 为 (2, 2)，dtype 为 int16。

（2）先预测下面四个结果的形状，再运行核对。逐项解释哪些轴被保留、移除或增加。

In [14]:
table = np.array([[10, 11, 12], [20, 21, 22]], dtype=np.int16)

# 先在此记录预测，再用原 shape (2, 3) 核对轴的变化。
print(table[0].shape)
print(table[0:1].shape)
print(table[:, 1, np.newaxis].shape)
print(table[2:5].shape)

(3,)
(1, 3)
(2, 1)
(0, 3)


（3）需要对下方 table 的最后两列制作临时修订版：只把修订版左上角改为 99，原数组必须保持不变。同时要求修订版仍是二维。从直接切片和“切片后 copy()”中选择写法，说明理由并检查共享关系。

In [15]:
table = np.array([[10, 11, 12], [20, 21, 22]], dtype=np.int16)

# 在此说明选择理由，创建修订版并修改。
# 检查：修订版为 (2, 2)，左上角为 99；table[0, 1] 仍为 11。
# 用 shares_memory() 检查修订版与原数组不共享元素内存。

（4）下面三个轴依次表示批次、观测和传感器。用省略号提取每次观测的最后一个传感器；再用切片把第二批前两次观测的最后两个传感器设为 0，其余位置保持不变。

In [16]:
batches = np.array([
    [[11, 12, 13, 14], [21, 22, 23, 24], [31, 32, 33, 34]],
    [[111, 112, 113, 114], [121, 122, 123, 124], [131, 132, 133, 134]],
], dtype=np.int16)

# 在此提取并打印，检查结果 shape 为 (2, 3)。
# 在此切片赋值，再打印第二批与原数组的 shape、dtype。
# 检查：第二批前两行为 [111 112 0 0]、[121 122 0 0]。
# 第三行及第一批保持不变；完整数组仍为 (2, 3, 4)，dtype 为 int16。

### 重点练习提示

对应第（3）题。先独立完成，再按需要查看提示。

（1）先判断切片是否隔离数据，再考虑二维形状是否保留。

（2）用全部行和最后两列取得区域；在修改前复制，并分别检查共享关系和原数组对应位置。

### 重点练习参考解析

对应第（3）题。

先取得 table[:, -2:]，再调用 copy()，得到独立的二维修订版。把修订版的 [0, 0] 改为 99 后，其两行为 [99, 12]、[21, 22]，形状为 (2, 2)，dtype 为 int16；原数组的 table[0, 1] 仍为 11。

基本切片是视图，直接修改会影响原数组；复制区域才能满足隔离要求。shares_memory 的结果应为 False，再结合原数组未变一起核对。只检查形状正确，不能证明修改已经隔离。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | [Indexing on ndarrays](https://numpy.org/doc/2.5/user/basics.indexing.html) 的 Single element indexing、Slicing and striding、Dimensional indexing tools、Assigning values to indexed arrays：位置与维数、切片、形状与类型约束；[Quickstart — Indexing, slicing and iterating](https://numpy.org/doc/2.5/user/quickstart.html#indexing-slicing-and-iterating)：步长、反向选择、多维索引和省略号；[Constants — newaxis](https://numpy.org/doc/2.5/reference/constants.html#numpy.newaxis)：None 别名与增轴示例；[Copies and views](https://numpy.org/doc/2.5/user/basics.copies.html) 的 View、Copy、Indexing operations：共享修改与 copy()；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html) 的定义与 Returns：共享元素内存的检查。 |
| Python 官方文档（Python 3.12） | [Built-in Types — Common Sequence Operations](https://docs.python.org/3.12/library/stdtypes.html#common-sequence-operations) 的注释 3～5：切片边界截取、空切片和正负步长；NumPy 对每个轴采用这些基本切片规则。 |